# Attention Graph Explorer

This notebook explores the bottom-up attention graph, the new DB-backed taxonomy, and how the two line up.

It is designed to answer:
- Did the monthly taxonomy backfill publish yet?
- Which candidate symbols are already covered by taxonomy rows?
- How does taxonomy change sector, industry, and peer-group interpretation inside the graph?
- What does the full candidate graph look like beyond one symbol's connected component?

The notebook is **graph-materialized-first** and **taxonomy-DB-first**:
- Graph artifacts come from the pipeline store when available, with an optional live rebuild fallback.
- Taxonomy rows come from `load_entity_taxonomy_frame()`, which checks Postgres before the materialized `entity_taxonomy_labels` dataset.

In [14]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

try:
    import networkx as nx
except Exception:
    nx = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 140)

In [15]:
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'data_access').exists() and (candidate / 'services').exists():
        ROOT = candidate
        break
else:
    fallback = Path('/home/azureuser/cloudfiles/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')
    if fallback.exists():
        ROOT = fallback
    else:
        raise RuntimeError('Could not locate the streamlit_alpaca_app project root from this notebook.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_access.layer import DataAccessLayer, _make_api
from services.pipeline_store import load_latest_dataset_frame
from services.entity_taxonomy import load_entity_taxonomy_frame
from services.attention_agentic import build_bottom_up_attention_artifacts, recompute_attention_candidate_graph
from services.attention_home_1d import build_attention_entity_master, resolve_macro_anchor_symbols, shortlist_attention_symbols_1d

ROOT


PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')

## Settings

In [16]:
FORCE_REFRESH = False
ALLOW_LIVE_REBUILD = True
GRAPH_EDGE_SOURCE = 'recompute'  # 'materialized' or 'recompute'
RESEARCH_SYMBOL_LIMIT = 40
FOCUS_SYMBOLS = []
SYMBOL = str(FOCUS_SYMBOLS[0]).upper().strip() if FOCUS_SYMBOLS else ''
COMPONENT_LIMIT = 12
GRAPH_LAYOUT_SEED = 7
NETWORK_BACKBONE_TOP_K = 0
NETWORK_BACKBONE_QUANTILE = 0.9
FOCUS_NETWORK_RADIUS = 1
NETWORK_LABEL_TOP_N = 10


## Load graph and taxonomy artifacts

In [17]:
DATASET_NAMES = [
    'attention_candidates_1d',
    'attention_candidate_graph',
    'attention_event_clusters_1d',
    'attention_claims',
    'attention_search_results',
    'attention_source_documents',
    'price_history',
    'us_equity_listings',
    'entity_taxonomy_labels',
]


def parse_jsonish(value, default=None):
    if value is None:
        return [] if default is None else default
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, (tuple, set)):
        return list(value)
    if hasattr(value, 'tolist') and not isinstance(value, (str, bytes)):
        converted = value.tolist()
        if isinstance(converted, (list, dict)):
            return converted
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return [] if default is None else default
    try:
        return json.loads(text)
    except Exception:
        return [] if default is None else default


def normalize_symbol(value):
    text = str(value or '').upper().strip()
    return '' if text == 'NAN' else text


def informative_text(value):
    text = str(value or '').strip()
    if not text or text.lower() == 'nan':
        return ''
    return text


def choose_label(primary, secondary='', default='Unknown'):
    for value in (primary, secondary):
        text = informative_text(value)
        if text and text != 'Unknown':
            return text
    return default


def to_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def load_materialized_graph_frames():
    frames = {}
    metadata = {}
    for dataset_name in DATASET_NAMES:
        frame, meta = load_latest_dataset_frame(dataset_name)
        frames[dataset_name] = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame()
        metadata[dataset_name] = meta
    required = ['attention_candidates_1d', 'attention_candidate_graph', 'attention_event_clusters_1d']
    ready = all(not frames[name].empty for name in required)
    return frames, metadata, ready


def rebuild_graph_frames_live(force_refresh=False):
    layer = DataAccessLayer.from_environment()
    if layer.cfg is None:
        raise RuntimeError('App config is unavailable, so the live attention graph cannot be rebuilt here.')

    equity_universe = layer._attention_home_equity_universe(force_refresh=force_refresh)

    holdings = []
    try:
        positions = layer.resolve_positions(force_refresh=force_refresh).payload
        if isinstance(positions, pd.DataFrame) and not positions.empty and 'symbol' in positions.columns:
            holdings = [
                str(value).upper().strip()
                for value in positions['symbol'].dropna().astype(str).tolist()
                if str(value).strip()
            ]
    except Exception:
        holdings = []

    equity_movers = layer.resolve_daily_movers(symbols=equity_universe, force_refresh=force_refresh).payload
    macro_anchor_symbols = resolve_macro_anchor_symbols(equity_universe)
    macro_movers = layer.resolve_daily_movers(symbols=macro_anchor_symbols, force_refresh=force_refresh).payload if macro_anchor_symbols else pd.DataFrame()
    movers = layer._combine_mover_frames(equity_movers, macro_movers, macro_anchor_symbols=macro_anchor_symbols)

    attention_parts = []
    for dataset_name in ('attention_feed', 'commodity_attention_feed'):
        try:
            resolved = layer.resolve_attention_feed(
                dataset_name=dataset_name,
                limit=80,
                horizons=['1d'],
                statuses=['active', 'cooling'],
                sensitivity='aggressive',
                force_refresh=force_refresh,
            ).payload
        except Exception:
            resolved = pd.DataFrame()
        if isinstance(resolved, pd.DataFrame) and not resolved.empty:
            attention_parts.append(resolved)
    attention_rows = pd.concat(attention_parts, ignore_index=True, sort=False) if attention_parts else pd.DataFrame()

    shortlist = shortlist_attention_symbols_1d(
        movers,
        holdings=holdings,
        attention_rows=attention_rows,
        max_count=100,
    )
    entity_master = build_attention_entity_master(shortlist)

    bars_by_symbol = {}
    if shortlist:
        try:
            bars_by_symbol = _make_api(layer.cfg).get_stock_bars(
                shortlist,
                start=datetime.now(timezone.utc) - pd.Timedelta(days=120),
                end=datetime.now(timezone.utc),
                timeframe='1Day',
                feed='iex',
            )
        except Exception:
            bars_by_symbol = {}

    research_symbols = shortlist[:RESEARCH_SYMBOL_LIMIT]
    news_payloads = {}
    context_payloads = {}
    for symbol in research_symbols:
        try:
            news_payloads[symbol] = layer.resolve_recent_news(symbol, days=3, limit=8, force_refresh=force_refresh).payload
        except Exception:
            news_payloads[symbol] = {'articles': pd.DataFrame(), 'fallback_summary': None, 'source': None}
        try:
            context_payloads[symbol] = layer.resolve_attention_context(symbol, force_refresh=force_refresh).payload
        except Exception:
            context_payloads[symbol] = {}

    filings_frame = layer._resolve_attention_edgar_filings(shortlist[:60], force_refresh=force_refresh)
    fred_summary_frame = pd.DataFrame()
    try:
        fred_summary_frame, _ = layer._pipeline_frame('fred_summary')
    except Exception:
        fred_summary_frame = pd.DataFrame()

    try:
        yield_curve_facts_frame = layer.resolve_yield_curve_facts_1d(force_refresh=force_refresh).payload
    except Exception:
        yield_curve_facts_frame = pd.DataFrame()

    artifacts = build_bottom_up_attention_artifacts(
        movers,
        attention_rows=attention_rows,
        bars_by_symbol=bars_by_symbol,
        news_payloads=news_payloads,
        context_payloads=context_payloads,
        entity_master=entity_master,
        holdings=holdings,
        generated_at_utc=pd.Timestamp.utcnow(),
        filings_frame=filings_frame,
        fred_summary_frame=fred_summary_frame,
        yield_curve_facts_frame=yield_curve_facts_frame,
        llm_client=None,
        embedding_client=None,
        top_events_limit=5,
        must_read_limit=10,
        unresolved_limit=5,
    )

    meta = {
        name: {
            'source': 'live_rebuild',
            'run_id': artifacts.home_payload.get('run_id', ''),
            'generated_at_utc': artifacts.home_payload.get('generated_at_utc', ''),
        }
        for name in DATASET_NAMES
    }
    return artifacts.frames, meta


frames, metadata, materialized_ready = load_materialized_graph_frames()
load_mode = 'materialized'
load_note = 'Loaded graph artifacts directly from the pipeline store.'

if not materialized_ready:
    if not ALLOW_LIVE_REBUILD:
        raise RuntimeError(
            'Materialized graph datasets are unavailable in this environment and live rebuild is disabled.'
        )
    frames, metadata = rebuild_graph_frames_live(force_refresh=FORCE_REFRESH)
    load_mode = 'live_rebuild'
    load_note = 'Materialized graph datasets were unavailable, so the notebook rebuilt the graph through the DAL/attention builder.'

for dataset_name in DATASET_NAMES:
    frames.setdefault(dataset_name, pd.DataFrame())
    metadata.setdefault(dataset_name, None)

load_summary = pd.DataFrame(
    [
        {
            'dataset_name': name,
            'rows': int(len(frames.get(name, pd.DataFrame()))),
            'columns': list(frames.get(name, pd.DataFrame()).columns),
            'meta': metadata.get(name),
        }
        for name in DATASET_NAMES
    ]
)

print(f'load_mode={load_mode}')
print(load_note)
load_summary

load_mode=materialized
Loaded graph artifacts directly from the pipeline store.


,dataset_name,rows,columns,meta
0,attention_candidates_1d,82,"[candidate_id, symbol, headline, source_label, peer_group_name, direction, change_pct, abs_change_pct, expected_move_pct, surprise_pct, ...","PipelineDataset(dataset_name='attention_candidates_1d', dataset_version_id='attention_candidates_1d__20260331T202021Z__c38b8d60', blob_p..."
1,attention_candidate_graph,63,"[run_id, asof_time_utc, left_candidate_id, right_candidate_id, left_symbol, right_symbol, edge_weight, edge_reasons_json]","PipelineDataset(dataset_name='attention_candidate_graph', dataset_version_id='attention_candidate_graph__20260331T202021Z__c38b8d60', bl..."
2,attention_event_clusters_1d,7,"[run_id, asof_time_utc, event_id, member_candidate_ids_json, anchor_candidate_ids_json, driver_symbols_json, beneficiary_symbols_json, l...","PipelineDataset(dataset_name='attention_event_clusters_1d', dataset_version_id='attention_event_clusters_1d__20260331T202021Z__c38b8d60'..."
3,attention_claims,14,"[claim_id, run_id, bundle_subject, claim_text, claim_type, claim_entities, supports_hypothesis, freshness_class, relevance_score, causal...","PipelineDataset(dataset_name='attention_claims', dataset_version_id='attention_claims__20260331T202021Z__c38b8d60', blob_path='datasets/..."
4,attention_search_results,132,"[run_id, asof_time_utc, candidate_id, query_id, provider, result_id, title, url, snippet, error_text, result_kind, source, published_at,...","PipelineDataset(dataset_name='attention_search_results', dataset_version_id='attention_search_results__20260331T202021Z__c38b8d60', blob..."
5,attention_source_documents,170,"[run_id, asof_time_utc, candidate_id, bundle_subject, document_id, source_kind, source_provider, source_authority_bucket, authority_rank...","PipelineDataset(dataset_name='attention_source_documents', dataset_version_id='attention_source_documents__20260331T202021Z__c38b8d60', ..."
6,us_equity_listings,6560,"[symbol, exchange, security_name, is_etf, is_test_issue, source_file]","PipelineDataset(dataset_name='us_equity_listings', dataset_version_id='us_equity_listings__20260329T075519Z__de6f7976', blob_path='datas..."
7,entity_taxonomy_labels,6559,"[symbol, exchange, security_name, listing_source, is_active, is_etf, asset_class, security_type, sector, industry, peer_group_name, peer...","PipelineDataset(dataset_name='entity_taxonomy_labels', dataset_version_id='entity_taxonomy_labels__20260329T075519Z__de6f7976', blob_pat..."


In [18]:
candidates = frames['attention_candidates_1d'].copy()
graph_edges = frames['attention_candidate_graph'].copy()
clusters = frames['attention_event_clusters_1d'].copy()
claims = frames['attention_claims'].copy()
search_results = frames['attention_search_results'].copy()
source_documents = frames['attention_source_documents'].copy()
price_history = frames['price_history'].copy()
listings = frames['us_equity_listings'].copy()
taxonomy_materialized = frames['entity_taxonomy_labels'].copy()

graph_edge_source = 'materialized'
graph_edge_note = 'Using the materialized attention_candidate_graph dataset.'
materialized_graph_edges = graph_edges.copy()

for frame in [candidates, listings, taxonomy_materialized]:
    if 'symbol' in frame.columns:
        frame['symbol'] = frame['symbol'].map(normalize_symbol)

for col in ['macro_exposure_tags', 'business_tags', 'macro_role_tags', 'business_role_tags']:
    if col in candidates.columns:
        candidates[col] = candidates[col].map(parse_jsonish)
    if col in taxonomy_materialized.columns:
        taxonomy_materialized[col] = taxonomy_materialized[col].map(parse_jsonish)

if 'edge_reasons_json' in graph_edges.columns:
    graph_edges['edge_reasons'] = graph_edges['edge_reasons_json'].map(parse_jsonish)
for col in ['left_symbol', 'right_symbol']:
    if col in graph_edges.columns:
        graph_edges[col] = graph_edges[col].map(normalize_symbol)

cluster_json_cols = [
    'member_candidate_ids_json',
    'anchor_candidate_ids_json',
    'driver_symbols_json',
    'beneficiary_symbols_json',
    'loser_symbols_json',
    'supporting_claim_ids_json',
    'event_facts_json',
]
for col in cluster_json_cols:
    if col in clusters.columns:
        clusters[col.replace('_json', '')] = clusters[col].map(parse_jsonish)

if 'claim_entities' in claims.columns:
    claims['claim_entities'] = claims['claim_entities'].map(parse_jsonish)
if 'claim_entities_json' in claims.columns:
    claims['claim_entities_json'] = claims['claim_entities_json'].map(parse_jsonish)
if 'evidence_chunk_ids' in claims.columns:
    claims['evidence_chunk_ids'] = claims['evidence_chunk_ids'].map(parse_jsonish)

try:
    taxonomy = load_entity_taxonomy_frame().copy()
    taxonomy_status = 'ok'
except Exception as exc:
    taxonomy = pd.DataFrame()
    taxonomy_status = f'{type(exc).__name__}: {exc}'

if not taxonomy.empty:
    if 'symbol' in taxonomy.columns:
        taxonomy['symbol'] = taxonomy['symbol'].map(normalize_symbol)
    for col in ['macro_role_tags', 'business_role_tags']:
        if col in taxonomy.columns:
            taxonomy[col] = taxonomy[col].map(parse_jsonish)

listing_cols = ['symbol', 'exchange', 'security_name', 'is_etf']
listing_lookup = listings[[col for col in listing_cols if col in listings.columns]].drop_duplicates('symbol') if not listings.empty and 'symbol' in listings.columns else pd.DataFrame(columns=listing_cols)

taxonomy_cols = [
    'symbol',
    'exchange',
    'security_name',
    'asset_class',
    'security_type',
    'sector',
    'industry',
    'peer_group_name',
    'peer_group_id',
    'source_of_truth',
    'label_provider',
    'label_confidence',
    'business_role_tags',
    'macro_role_tags',
    'country',
    'updated_at_utc',
]
taxonomy_lookup = taxonomy[[col for col in taxonomy_cols if col in taxonomy.columns]].copy() if not taxonomy.empty else pd.DataFrame(columns=taxonomy_cols)
if not taxonomy_lookup.empty:
    taxonomy_lookup = taxonomy_lookup.rename(columns={col: f'taxonomy_{col}' for col in taxonomy_lookup.columns if col != 'symbol'})

candidate_graph = candidates.copy()
if 'symbol' in candidate_graph.columns:
    candidate_graph['symbol_upper'] = candidate_graph['symbol'].map(normalize_symbol)
else:
    candidate_graph['symbol_upper'] = ''

if not listing_lookup.empty:
    candidate_graph = candidate_graph.merge(
        listing_lookup.rename(columns={'symbol': 'symbol_upper'}),
        on='symbol_upper',
        how='left',
        suffixes=('', '_listing'),
    )
if not taxonomy_lookup.empty:
    candidate_graph = candidate_graph.merge(
        taxonomy_lookup.rename(columns={'symbol': 'symbol_upper'}),
        on='symbol_upper',
        how='left',
    )

for col in ['taxonomy_business_role_tags', 'taxonomy_macro_role_tags']:
    if col not in candidate_graph.columns:
        candidate_graph[col] = [[] for _ in range(len(candidate_graph))]
    else:
        candidate_graph[col] = candidate_graph[col].map(parse_jsonish)

for field in ['sector', 'industry', 'peer_group_name', 'peer_group_id', 'asset_class', 'security_type', 'country']:
    candidate_graph[f'effective_{field}'] = [
        choose_label(row.get(f'taxonomy_{field}', ''), row.get(field, ''))
        for _, row in candidate_graph.iterrows()
    ]

if 'taxonomy_source_of_truth' in candidate_graph.columns:
    candidate_graph['effective_source_of_truth'] = candidate_graph['taxonomy_source_of_truth'].fillna('attention_candidates_1d_only')
else:
    candidate_graph['effective_source_of_truth'] = 'attention_candidates_1d_only'

candidate_graph['taxonomy_present'] = candidate_graph.get('taxonomy_source_of_truth', pd.Series(index=candidate_graph.index, dtype=object)).notna()

if GRAPH_EDGE_SOURCE == 'recompute' and not candidate_graph.empty:
    candidate_graph, graph_edges = recompute_attention_candidate_graph(
        candidate_graph,
        claims_frame=claims,
        price_history_frame=price_history,
        run_id='notebook_recompute',
        asof_time_utc=pd.Timestamp.utcnow(),
    )
    graph_edge_source = 'recompute'
    if isinstance(price_history, pd.DataFrame) and not price_history.empty:
        graph_edge_note = 'Recomputed graph edges from the current notebook candidates and claims using the local attention graph builder, with historical return correlation available for sparse pairs.'
    else:
        graph_edge_note = 'Recomputed graph edges from the current notebook candidates and claims using the local attention graph builder. Historical-correlation support is unavailable because price_history is missing.'
elif GRAPH_EDGE_SOURCE != 'materialized':
    raise ValueError(f'Unsupported GRAPH_EDGE_SOURCE={GRAPH_EDGE_SOURCE}')

if 'edge_reasons_json' in graph_edges.columns:
    graph_edges['edge_reasons'] = graph_edges['edge_reasons_json'].map(parse_jsonish)
for col in ['left_symbol', 'right_symbol']:
    if col in graph_edges.columns:
        graph_edges[col] = graph_edges[col].map(normalize_symbol)


def resolve_focus_symbol(candidate_frame, edge_frame, requested_symbols, default_symbol=''):
    candidate_symbols = [normalize_symbol(value) for value in candidate_frame.get('symbol_upper', pd.Series(dtype=str)).tolist()]
    candidate_symbols = [value for value in candidate_symbols if value]
    candidate_symbol_set = set(candidate_symbols)
    if not candidate_symbol_set:
        return '', ''

    requested = [normalize_symbol(value) for value in requested_symbols if normalize_symbol(value)]
    for value in requested:
        if value in candidate_symbol_set:
            return value, ''

    degree_map = {}
    if isinstance(edge_frame, pd.DataFrame) and not edge_frame.empty:
        for _, row in edge_frame.iterrows():
            left = normalize_symbol(row.get('left_symbol'))
            right = normalize_symbol(row.get('right_symbol'))
            if left:
                degree_map[left] = degree_map.get(left, 0) + 1
            if right:
                degree_map[right] = degree_map.get(right, 0) + 1

    score_map = {}
    if 'candidate_score' in candidate_frame.columns:
        for _, row in candidate_frame.iterrows():
            score_map[normalize_symbol(row.get('symbol_upper'))] = to_float(row.get('candidate_score'))

    ranked = sorted(
        candidate_symbol_set,
        key=lambda value: (-degree_map.get(value, 0), -score_map.get(value, 0.0), value),
    )
    selected = ranked[0] if ranked else normalize_symbol(default_symbol)
    if requested and selected:
        return selected, f'Requested focus symbol(s) were not present in the current graph; using {selected} instead.'
    if selected and not requested:
        return selected, f'Using {selected} as the default focus symbol.'
    return selected, ''

focus_symbols = [normalize_symbol(value) for value in FOCUS_SYMBOLS if normalize_symbol(value)]
symbol, focus_symbol_note = resolve_focus_symbol(candidate_graph, graph_edges, focus_symbols, SYMBOL)
if focus_symbol_note:
    print(focus_symbol_note)

print('candidates', candidates.shape)
print('graph_edges', graph_edges.shape)
print('graph_edges_materialized', materialized_graph_edges.shape)
print('clusters', clusters.shape)
print('claims', claims.shape)
print('price_history', price_history.shape)
print('listings', listings.shape)
print('taxonomy_materialized', taxonomy_materialized.shape)
print('taxonomy_db_first', taxonomy.shape)
print(f'taxonomy_status={taxonomy_status}')
print(f'graph_edge_source={graph_edge_source}')
print(graph_edge_note)


Using KOD as the default focus symbol.
candidates (82, 48)
graph_edges (78, 9)
graph_edges_materialized (63, 8)
clusters (7, 20)
claims (14, 18)
listings (6560, 6)
taxonomy_materialized (6559, 26)
taxonomy_db_first (6559, 26)
taxonomy_status=ok
graph_edge_source=recompute
Recomputed graph edges from the current notebook candidates and claims using the local attention graph builder.


## Taxonomy coverage

In [19]:
candidate_symbols = sorted(set(candidate_graph['symbol_upper'])) if not candidate_graph.empty else []
listing_symbols = sorted(set(listings['symbol'])) if not listings.empty and 'symbol' in listings.columns else []
taxonomy_symbols = sorted(set(taxonomy['symbol'])) if not taxonomy.empty and 'symbol' in taxonomy.columns else []
covered_candidate_symbols = sorted(set(candidate_graph.loc[candidate_graph['taxonomy_present'], 'symbol_upper'])) if not candidate_graph.empty else []
missing_candidate_symbols = sorted(set(candidate_symbols) - set(covered_candidate_symbols))
listing_taxonomy_coverage = len(set(listing_symbols) & set(taxonomy_symbols))

coverage_summary = pd.DataFrame(
    [
        {
            'metric': 'candidate_symbols',
            'value': len(candidate_symbols),
        },
        {
            'metric': 'candidate_symbols_with_taxonomy',
            'value': len(covered_candidate_symbols),
        },
        {
            'metric': 'candidate_taxonomy_coverage_pct',
            'value': round(100.0 * len(covered_candidate_symbols) / max(len(candidate_symbols), 1), 1),
        },
        {
            'metric': 'listing_symbols',
            'value': len(listing_symbols),
        },
        {
            'metric': 'listing_symbols_with_taxonomy',
            'value': listing_taxonomy_coverage,
        },
        {
            'metric': 'listing_taxonomy_coverage_pct',
            'value': round(100.0 * listing_taxonomy_coverage / max(len(listing_symbols), 1), 1),
        },
        {
            'metric': 'taxonomy_materialized_rows',
            'value': int(len(taxonomy_materialized)),
        },
        {
            'metric': 'taxonomy_db_first_rows',
            'value': int(len(taxonomy)),
        },
    ]
)

display(coverage_summary)

if not taxonomy.empty:
    source_counts = taxonomy['source_of_truth'].fillna('unknown').value_counts().rename_axis('source_of_truth').reset_index(name='rows')
    sector_counts = taxonomy['sector'].fillna('Unknown').value_counts().head(20).rename_axis('sector').reset_index(name='rows')
    industry_counts = taxonomy['industry'].fillna('Unknown').value_counts().head(20).rename_axis('industry').reset_index(name='rows')
    display(source_counts)
    display(sector_counts)
    display(industry_counts)
else:
    print('No taxonomy rows are currently visible through DB or materialized datasets.')
    print('The notebook is wired for the new taxonomy, but the monthly backfill has not published labels yet.')

pd.DataFrame({'candidate_symbols_waiting_for_taxonomy': missing_candidate_symbols[:60]})

,metric,value
0,candidate_symbols,82.0
1,candidate_symbols_with_taxonomy,82.0
2,candidate_taxonomy_coverage_pct,100.0
3,listing_symbols,6560.0
4,listing_symbols_with_taxonomy,6559.0
5,listing_taxonomy_coverage_pct,100.0
6,taxonomy_materialized_rows,6559.0
7,taxonomy_db_first_rows,6559.0


,source_of_truth,rows
0,llm_taxonomy,6559


,sector,rows
0,Financials,1016
1,Health Care,990
2,Information Technology,771
3,Broad Market,718
4,Industrials,650
5,Consumer Discretionary,518
6,Credit,374
7,Materials,278
8,Energy,261
9,Communication Services,261


,industry,rows
0,Biotechnology,553
1,Medical Devices,121
2,Regional Bank,101
3,regional_bank,55
4,Asset Management,44
5,SPAC,43
6,Leveraged Single-Stock ETF,41
7,Special Purpose Acquisition Company,38
8,Regional Banks,33
9,regional bank,29


,candidate_symbols_waiting_for_taxonomy


## Candidate taxonomy view

In [20]:
candidate_view_cols = [
    'candidate_id',
    'symbol',
    'exchange',
    'security_name',
    'change_pct',
    'candidate_score',
    'sector',
    'industry',
    'taxonomy_sector',
    'taxonomy_industry',
    'effective_sector',
    'effective_industry',
    'effective_peer_group_id',
    'taxonomy_source_of_truth',
    'taxonomy_label_confidence',
    'business_tags',
    'taxonomy_business_role_tags',
]
candidate_view_cols = [col for col in candidate_view_cols if col in candidate_graph.columns]

candidate_view = candidate_graph[[*candidate_view_cols, *[col for col in ['taxonomy_present'] if col in candidate_graph.columns and col not in candidate_view_cols]]].copy()
sort_by = [col for col in ['taxonomy_present', 'candidate_score', 'change_pct'] if col in candidate_view.columns]
if sort_by:
    candidate_view = candidate_view.sort_values(sort_by, ascending=[False] * len(sort_by))

candidate_view[candidate_view_cols].head(50)

,candidate_id,symbol,exchange,security_name,change_pct,candidate_score,sector,industry,taxonomy_sector,taxonomy_industry,effective_sector,effective_industry,effective_peer_group_id,taxonomy_source_of_truth,taxonomy_label_confidence,business_tags,taxonomy_business_role_tags
0,candidate::FLY,FLY,NASDAQ,Firefly Aerospace Inc. - Common Stock,19.24,147.1,Industrials,Aerospace & Defense,Industrials,Space Launch Services,Industrials,Space Launch Services,Space Launch Services,llm_taxonomy,low,[],[launch_service_provider_rocket_manufacturer]
1,candidate::CDE,CDE,NYSE,"Coeur Mining, Inc. Common Stock",12.36,139.8,Materials,Precious Metals Mining,Materials,Precious Metals Mining,Materials,Precious Metals Mining,Precious Metals Mining,llm_taxonomy,high,[commodity],[commodity]
2,candidate::AMPX,AMPX,NYSE,"Amprius Technologies, Inc. Common Stock",13.10,138.2,Industrials,Advanced Batteries,Industrials,Advanced Lithium Battery Technology,Industrials,Advanced Lithium Battery Technology,Advanced Lithium Battery Technology,llm_taxonomy,medium,[],[battery_developer_advanced_battery_manufacturer]
3,candidate::RBLX,RBLX,NYSE,Roblox Corporation Class A Common Stock,10.03,136.4,Communication Services,Online Gaming Platform,Communication Services,Online Gaming Platform,Communication Services,Online Gaming Platform,Online Gaming Platform,llm_taxonomy,high,[social_media_entertainment],[social_media_entertainment]
4,candidate::FSLY,FSLY,NASDAQ,"Fastly, Inc. - Class A Common Stock",10.88,125.2,Information Technology,Cloud Infrastructure,Information Technology,Edge Cloud & Content Delivery Network Services,Information Technology,Edge Cloud & Content Delivery Network Services,Edge Cloud & Content Delivery Network Services,llm_taxonomy,high,[],[content_delivery_network_edge_cloud_platform]
5,candidate::MRVL,MRVL,NASDAQ,"Marvell Technology, Inc. - Common Stock",12.56,124.6,Information Technology,Semiconductors,Information Technology,Semiconductor Design,Information Technology,Semiconductor Design,Semiconductor Design,llm_taxonomy,high,[],[fabless_semiconductor_designer_data_center_chip_supplier]
6,candidate::KOD,KOD,NASDAQ,Kodiak Sciences Inc - Common Stock,16.49,122.0,Health Care,Biotechnology,Health Care,Biotechnology,Health Care,Biotechnology,Biotechnology,llm_taxonomy,high,[healthcare_life_sciences],[healthcare_life_sciences]
7,candidate::SLVR,SLVR,NASDAQ,Sprott Silver Miners & Physical Silver ETF,-28.32,119.3,Commodities,Silver & Silver Miners ETF,Commodities,Silver & Silver Miners ETF,Commodities,Silver & Silver Miners ETF,Silver & Silver Miners ETF,llm_taxonomy,high,[commodity],[commodity]
8,candidate::UUUG,UUUG,NASDAQ,Leverage Shares 2X Long UUUU Daily ETF,-43.59,117.8,Commodities,Leveraged Uranium ETF,Commodities,Leveraged Uranium ETF,Commodities,Leveraged Uranium ETF,Leveraged Uranium ETF,llm_taxonomy,medium,[commodity],[commodity]
9,candidate::TXXS,TXXS,NASDAQ,21Shares 2x Long Sui ETF,-74.94,116.2,Commodities,Crypto Asset ETF,Commodities,Crypto Asset ETF,Commodities,Crypto Asset ETF,Crypto Asset ETF,llm_taxonomy,medium,[commodity],[commodity]


## Edge table

In [21]:
candidate_lookup_cols = [
    'candidate_id',
    'symbol',
    'change_pct',
    'candidate_score',
    'effective_sector',
    'effective_industry',
    'effective_peer_group_id',
    'effective_source_of_truth',
    'macro_exposure_tags',
    'business_tags',
    'taxonomy_business_role_tags',
]
candidate_lookup_cols = [col for col in candidate_lookup_cols if col in candidate_graph.columns]
candidate_lookup = candidate_graph[candidate_lookup_cols].copy()

left_lookup = candidate_lookup.add_prefix('left_')
right_lookup = candidate_lookup.add_prefix('right_')

edges_enriched = graph_edges.merge(
    left_lookup,
    left_on='left_candidate_id',
    right_on='left_candidate_id',
    how='left',
).merge(
    right_lookup,
    left_on='right_candidate_id',
    right_on='right_candidate_id',
    how='left',
)
for canonical, options in {
    'left_symbol': ['left_symbol', 'left_symbol_x', 'left_symbol_y'],
    'right_symbol': ['right_symbol', 'right_symbol_x', 'right_symbol_y'],
}.items():
    available = [col for col in options if col in edges_enriched.columns]
    if not available:
        continue
    series = edges_enriched[available[0]]
    for col in available[1:]:
        series = series.combine_first(edges_enriched[col])
    edges_enriched[canonical] = series.map(normalize_symbol)

edge_view_cols = [
    'left_symbol',
    'right_symbol',
    'edge_weight',
    'edge_reasons',
    'left_effective_sector',
    'right_effective_sector',
    'left_effective_industry',
    'right_effective_industry',
    'left_effective_peer_group_id',
    'right_effective_peer_group_id',
    'left_change_pct',
    'right_change_pct',
]
edge_view_cols = [col for col in edge_view_cols if col in edges_enriched.columns]
edges_enriched[edge_view_cols].sort_values('edge_weight', ascending=False).head(50)

,left_symbol,right_symbol,edge_weight,edge_reasons,left_effective_sector,right_effective_sector,left_effective_industry,right_effective_industry,left_effective_peer_group_id,right_effective_peer_group_id,left_change_pct,right_change_pct
30,INSM,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,6.45,-0.27
38,ERAS,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,5.72,-0.27
55,VRDN,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,4.19,-0.27
12,KOD,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,16.49,-0.27
44,DNTH,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,6.23,-0.27
62,XENE,TERN,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,4.52,-0.27
73,TERN,ADMA,0.760,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,-0.27,0.77
28,INSM,VRDN,0.740,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,6.45,4.19
27,INSM,DNTH,0.740,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,6.45,6.23
26,INSM,ERAS,0.740,"[taxonomy_peer, tags]",Health Care,Health Care,Biotechnology,Biotechnology,Biotechnology,Biotechnology,6.45,5.72


## Cluster summary

In [22]:
def cluster_member_symbols(event_facts):
    facts = event_facts if isinstance(event_facts, dict) else {}
    members = facts.get('members') or []
    return [normalize_symbol(item.get('symbol', '')) for item in members if normalize_symbol(item.get('symbol', ''))]


cluster_summary = clusters.copy()
if 'event_facts' in cluster_summary.columns:
    cluster_summary['member_symbols'] = cluster_summary['event_facts'].map(cluster_member_symbols)
else:
    cluster_summary['member_symbols'] = [[] for _ in range(len(cluster_summary))]

cluster_summary['member_count'] = cluster_summary['member_symbols'].map(len)
cluster_summary['member_symbols_text'] = cluster_summary['member_symbols'].map(lambda items: ', '.join(items[:12]))

cluster_view_cols = [
    'event_id',
    'event_type',
    'cause_status',
    'event_score',
    'member_count',
    'member_symbols_text',
    'driver_symbols',
    'beneficiary_symbols',
    'loser_symbols',
]
cluster_view_cols = [col for col in cluster_view_cols if col in cluster_summary.columns]
cluster_summary[cluster_view_cols].sort_values('event_score', ascending=False).head(20)

,event_id,event_type,cause_status,event_score,member_count,member_symbols_text,driver_symbols,beneficiary_symbols,loser_symbols
2,cluster-03-956e0135a3,glow2,supported,629.2,8,"KOD, INSM, ERAS, DNTH, VRDN, XENE, TERN, ADMA","[KOD, INSM, ERAS, DNTH, VRDN, XENE]","[KOD, INSM, ERAS, DNTH, VRDN, XENE]",[TERN]
0,cluster-01-5e0a490707,commodity,unresolved,494.9,6,"CDE, FCX, B, DOW, LYB, LIN","[CDE, FCX, B]","[CDE, FCX, B]","[DOW, LYB, LIN]"
3,cluster-04-f80ffbeea5,commodity,unresolved,361.3,3,"SLVR, UUUG, TXXS","[SLVR, UUUG, TXXS]",[],"[SLVR, UUUG, TXXS]"
6,cluster-07-85d0a6e0e4,commodity,unresolved,304.2,5,"LNG, CNQ, PSX, BP, MPC","[LNG, CNQ, PSX, BP, MPC]",[],"[LNG, CNQ, PSX, BP, MPC]"
4,cluster-05-0984264e58,mobility,unresolved,270.4,3,"UAL, DAL, CAR","[UAL, DAL, CAR]","[UAL, DAL, CAR]",[]
5,cluster-06-deb7f05f83,commerce,unresolved,269.9,3,"COIN, CRCL, HOOD","[COIN, CRCL, HOOD]","[COIN, CRCL, HOOD]",[]
1,cluster-02-d25bc087a8,entertainment,unresolved,249.6,2,"RBLX, META","[RBLX, META]","[RBLX, META]",[]


## Symbol inspector

In [23]:
symbol_row = candidate_graph[candidate_graph['symbol_upper'] == symbol].copy() if not candidate_graph.empty else pd.DataFrame()
taxonomy_row = taxonomy[taxonomy['symbol'] == symbol].copy() if not taxonomy.empty and 'symbol' in taxonomy.columns else pd.DataFrame()
neighbor_edges = edges_enriched[
    (edges_enriched['left_symbol'].astype(str).str.upper() == symbol)
    | (edges_enriched['right_symbol'].astype(str).str.upper() == symbol)
].copy() if not edges_enriched.empty else pd.DataFrame()


def extract_other_symbol(row, target):
    left = normalize_symbol(row.get('left_symbol', ''))
    right = normalize_symbol(row.get('right_symbol', ''))
    return right if left == target else left


if not neighbor_edges.empty:
    neighbor_edges['other_symbol'] = neighbor_edges.apply(lambda row: extract_other_symbol(row, symbol), axis=1)
    neighbor_edges = neighbor_edges.sort_values('edge_weight', ascending=False)

cluster_hits = cluster_summary[cluster_summary['member_symbols'].map(lambda items: symbol in set(items))].copy() if not cluster_summary.empty else pd.DataFrame()
claim_subject = claims.get('bundle_subject', pd.Series(dtype=str)).astype(str).str.upper() if not claims.empty else pd.Series(dtype=str)
claim_hits = claims[claim_subject == symbol].copy() if not claims.empty else pd.DataFrame()

symbol_view_cols = [
    'candidate_id',
    'symbol',
    'exchange',
    'security_name',
    'change_pct',
    'candidate_score',
    'effective_sector',
    'effective_industry',
    'effective_peer_group_id',
    'taxonomy_source_of_truth',
    'taxonomy_label_confidence',
    'business_tags',
    'taxonomy_business_role_tags',
]

taxonomy_view_cols = [
    'symbol',
    'exchange',
    'security_name',
    'sector',
    'industry',
    'peer_group_id',
    'source_of_truth',
    'label_provider',
    'label_confidence',
    'business_role_tags',
    'macro_role_tags',
    'updated_at_utc',
]

if not symbol:
    print('No focus symbol is available for inspection.')
elif symbol_row.empty:
    print(f'{symbol} is not present in the current attention candidate set.')
else:
    display(symbol_row[[col for col in symbol_view_cols if col in symbol_row.columns]])

if not symbol:
    print('No taxonomy row can be shown because no focus symbol is selected.')
elif taxonomy_row.empty:
    print(f'No taxonomy row is currently loaded for {symbol}.')
else:
    display(taxonomy_row[[col for col in taxonomy_view_cols if col in taxonomy_row.columns]])

display(neighbor_edges[[col for col in ['other_symbol', 'edge_weight', 'edge_reasons', 'left_effective_sector', 'right_effective_sector', 'left_change_pct', 'right_change_pct'] if col in neighbor_edges.columns]].head(20))
display(cluster_hits[cluster_view_cols])
display(claim_hits[[col for col in ['claim_text', 'claim_type', 'supports_hypothesis', 'causal_score', 'confidence_score', 'is_same_day', 'claim_entities'] if col in claim_hits.columns]].sort_values(['confidence_score', 'causal_score'], ascending=False).head(20))


,candidate_id,symbol,exchange,security_name,change_pct,candidate_score,effective_sector,effective_industry,effective_peer_group_id,taxonomy_source_of_truth,taxonomy_label_confidence,business_tags,taxonomy_business_role_tags
6,candidate::KOD,KOD,NASDAQ,Kodiak Sciences Inc - Common Stock,16.49,122.0,Health Care,Biotechnology,Biotechnology,llm_taxonomy,high,[healthcare_life_sciences],[healthcare_life_sciences]


,symbol,exchange,security_name,sector,industry,peer_group_id,source_of_truth,label_provider,label_confidence,business_role_tags,macro_role_tags,updated_at_utc
3310,KOD,NASDAQ,Kodiak Sciences Inc - Common Stock,Health Care,Biotechnology,Biotechnology,llm_taxonomy,llm,high,[healthcare_life_sciences],[],2026-03-29 00:09:04.974382+00:00


,other_symbol,edge_weight,edge_reasons,left_effective_sector,right_effective_sector,left_change_pct,right_change_pct
12,TERN,0.76,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,-0.27
7,INSM,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,6.45
8,ERAS,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,5.72
9,DNTH,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,6.23
10,VRDN,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,4.19
11,XENE,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,4.52
13,ADMA,0.74,"[taxonomy_peer, tags]",Health Care,Health Care,16.49,0.77


,event_id,event_type,cause_status,event_score,member_count,member_symbols_text,driver_symbols,beneficiary_symbols,loser_symbols
2,cluster-03-956e0135a3,glow2,supported,629.2,8,"KOD, INSM, ERAS, DNTH, VRDN, XENE, TERN, ADMA","[KOD, INSM, ERAS, DNTH, VRDN, XENE]","[KOD, INSM, ERAS, DNTH, VRDN, XENE]",[TERN]


,claim_text,claim_type,supports_hypothesis,causal_score,confidence_score,is_same_day,claim_entities
9,"Kodiak Sciences reported positive topline results from its second Phase 3 GLOW2 study in diabetic retinopathy, showing Zenkuda (tarcocim...",clinical_trial_result,company_specific,0.9,0.92,True,"[Kodiak Sciences, Zenkuda, tarcocimab tedromer, GLOW2]"
10,Kodiak Sciences appears to be decoupling from biotech peers following the positive topline GLOW2 Phase 3 results for Zenkuda in diabetic...,market_reaction,company_specific,0.6,0.55,False,"[Kodiak Sciences, Zenkuda]"


## Full graph overview

In [24]:
from pathlib import Path
import sys
import networkx as nx
import pandas as pd

if 'ROOT' not in globals():
    ROOT = Path.cwd().resolve()
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / 'data_access').exists() and (candidate / 'services').exists():
            ROOT = candidate
            break
    else:
        fallback = Path('/home/azureuser/cloudfiles/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')
        if fallback.exists():
            ROOT = fallback
        else:
            raise RuntimeError('Could not locate the streamlit_alpaca_app project root from this graph cell.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

missing_state = [
    name
    for name in ['candidate_graph', 'graph_edges', 'NETWORK_BACKBONE_TOP_K', 'NETWORK_BACKBONE_QUANTILE', 'COMPONENT_LIMIT']
    if name not in globals()
]
if missing_state:
    raise RuntimeError(
        'Missing notebook state: '
        + ', '.join(missing_state)
        + '. Run the setup/data cells through the graph-frame build cell before running the graph summary cells.'
    )

from services.attention_graph_network import (
    build_attention_candidate_network,
    build_network_backbone,
    connected_candidate_subgraph,
    expand_network_real_paths,
    network_graph_summary,
)
from services.attention_graph_topology import build_attention_topology_graph

candidate_network = build_attention_candidate_network(candidate_graph, graph_edges)
connected_network = connected_candidate_subgraph(candidate_network)
backbone_network = build_network_backbone(
    candidate_network,
    per_node_k=NETWORK_BACKBONE_TOP_K,
    keep_quantile=NETWORK_BACKBONE_QUANTILE,
)
topology_network = build_attention_topology_graph(
    candidate_graph,
    clusters,
    universe_frame=taxonomy,
)
plot_network = expand_network_real_paths(backbone_network, topology_network)
display_network = connected_candidate_subgraph(plot_network)
path_nodes = {
    node: attrs
    for node, attrs in display_network.nodes(data=True)
    if attrs.get('is_intermediate_path')
}
plot_connected_components = (
    nx.number_connected_components(display_network)
    if display_network.number_of_nodes()
    else 0
)

network_summary = pd.DataFrame(
    [
        {
            **network_graph_summary(candidate_network),
            'backbone_edges': int(backbone_network.number_of_edges()),
            'topology_nodes': int(topology_network.number_of_nodes()),
            'topology_edges': int(topology_network.number_of_edges()),
            'plot_nodes': int(display_network.number_of_nodes()),
            'plot_edges': int(display_network.number_of_edges()),
            'path_nodes': int(len(path_nodes)),
            'plot_connected_components': int(plot_connected_components),
        }
    ]
)

component_rows = []
for index, symbols in enumerate(sorted(nx.connected_components(connected_network), key=len, reverse=True), start=1):
    subgraph = connected_network.subgraph(symbols)
    sector_counts = pd.Series(
        [candidate_network.nodes[node].get('sector', 'Unknown') or 'Unknown' for node in subgraph.nodes()]
    ).value_counts()
    component_rows.append(
        {
            'component_id': index,
            'node_count': subgraph.number_of_nodes(),
            'edge_count': subgraph.number_of_edges(),
            'density': round(nx.density(subgraph), 3),
            'top_sectors': ', '.join(sector_counts.head(4).index.tolist()),
            'sample_symbols': ', '.join(sorted(list(symbols))[:COMPONENT_LIMIT]),
        }
    )

path_rows = [
    {
        'path_node': node,
        'label': attrs.get('symbol', node),
        'topology_type': attrs.get('topology_node_type', attrs.get('node_type', 'path')),
        'sector': attrs.get('topology_sector', attrs.get('sector', 'Unknown')),
        'peer_group': attrs.get('topology_peer_group', attrs.get('peer_group', 'Unknown')),
        'industry': attrs.get('topology_industry', attrs.get('industry', 'Unknown')),
    }
    for node, attrs in sorted(path_nodes.items(), key=lambda item: (str(item[1].get('topology_node_type', '')), item[0]))
]

display(network_summary)
display(pd.DataFrame(component_rows).head(20))
if path_rows:
    display(pd.DataFrame(path_rows).head(30))



ImportError: cannot import name 'expand_network_bridge_nodes' from 'services.attention_graph_network' (/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/streamlit_alpaca_app/services/attention_graph_network.py)

## Graph plots

In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd

if 'ROOT' not in globals():
    ROOT = Path.cwd().resolve()
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / 'data_access').exists() and (candidate / 'services').exists():
            ROOT = candidate
            break
    else:
        fallback = Path('/home/azureuser/cloudfiles/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')
        if fallback.exists():
            ROOT = fallback
        else:
            raise RuntimeError('Could not locate the streamlit_alpaca_app project root from this graph cell.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

missing_state = [
    name
    for name in [
        'candidate_graph',
        'graph_edges',
        'clusters',
        'taxonomy',
        'NETWORK_BACKBONE_TOP_K',
        'NETWORK_BACKBONE_QUANTILE',
        'GRAPH_LAYOUT_SEED',
        'NETWORK_LABEL_TOP_N',
    ]
    if name not in globals()
]
if missing_state:
    raise RuntimeError(
        'Missing notebook state: '
        + ', '.join(missing_state)
        + '. Run the setup/data cells through the graph-frame build cell before running the graph plot cells.'
    )

import services.attention_graph_network as attention_graph_network
import services.attention_graph_topology as attention_graph_topology

# Always reload graph modules so notebook plotting picks up latest code changes without kernel restart.
attention_graph_network = importlib.reload(attention_graph_network)
attention_graph_topology = importlib.reload(attention_graph_topology)

plot_candidate_graph = candidate_graph.copy()
if isinstance(plot_candidate_graph, pd.DataFrame) and not plot_candidate_graph.empty:
    if 'symbol_upper' not in plot_candidate_graph.columns and 'symbol' in plot_candidate_graph.columns:
        plot_candidate_graph['symbol_upper'] = (
            plot_candidate_graph['symbol'].astype(str).str.upper().str.strip()
        )
    if 'security_name' not in plot_candidate_graph.columns:
        plot_candidate_graph['security_name'] = ''

    if isinstance(taxonomy, pd.DataFrame) and not taxonomy.empty and {'symbol', 'security_name'}.issubset(taxonomy.columns):
        name_map = (
            taxonomy[['symbol', 'security_name']]
            .copy()
            .assign(symbol=lambda frame: frame['symbol'].astype(str).str.upper().str.strip())
            .dropna(subset=['symbol'])
            .drop_duplicates(subset=['symbol'], keep='first')
            .set_index('symbol')['security_name']
            .astype(str)
        )
        existing = plot_candidate_graph['security_name'].astype(str).str.strip()
        missing_mask = existing.eq('') | existing.str.lower().eq('nan')
        if 'symbol_upper' in plot_candidate_graph.columns:
            plot_candidate_graph.loc[missing_mask, 'security_name'] = (
                plot_candidate_graph.loc[missing_mask, 'symbol_upper'].map(name_map).fillna('')
            )

# Recompute every run to avoid stale cached network objects from earlier notebook executions.
candidate_network = attention_graph_network.build_attention_candidate_network(plot_candidate_graph, graph_edges)
connected_network = attention_graph_network.connected_candidate_subgraph(candidate_network)
backbone_network = attention_graph_network.build_network_backbone(
    candidate_network,
    per_node_k=NETWORK_BACKBONE_TOP_K,
    keep_quantile=NETWORK_BACKBONE_QUANTILE,
)
topology_network = attention_graph_topology.build_attention_topology_graph(
    plot_candidate_graph,
    clusters,
    universe_frame=taxonomy,
)
plot_network = attention_graph_network.expand_network_real_paths(backbone_network, topology_network)

label_top_n = max(int(NETWORK_LABEL_TOP_N), 18)

display(
    attention_graph_network.plot_attention_candidate_network(
        plot_network,
        title='Global attention network (backbone + real topology paths)',
        height=900,
        seed=GRAPH_LAYOUT_SEED,
        show_labels='auto',
        label_top_n=label_top_n,
        show_isolates=False,
        show_component_labels=True,
    )
)
